# Chapter 88 - Saving the Model, Tokenizer, and Config

A reusable language model is a package of compatible code, tensor state, tokenizer rules, and configuration.

This chapter saves that package, reconstructs it from fresh objects, and verifies an exact inference round trip.

## Learning goals

By the end of this chapter, you will be able to:

- save model state with `torch.save`;

- serialize a character tokenizer and validated configurations as JSON;

- load tensor state on CPU with restricted `torch.load` behavior;

- recreate model architecture from saved configuration;

- verify identical tokenizer IDs, state tensors, logits, and seeded generation;

- demonstrate why a different token-to-ID mapping corrupts model input; and

- distinguish an inference package from a checkpoint that can resume training exactly.

## Package anatomy

This notebook creates four files:

- `model_state.pt` stores the model's state dictionary;

- `tokenizer.json` stores the exact token-to-ID mapping and tokenizer format version;

- `model_config.json` stores the architecture name and dimensions; and

- `training_config.json` documents the training run.

The Python class definitions still matter because a state dictionary does not contain executable model architecture code.

The package is sufficient for inference only while compatible model and tokenizer code remain available.

## Terms and corrections

**Serialization** converts state into a representation that can be written to storage.

**Deserialization** reconstructs usable state from stored data.

A **state dictionary** maps module names to parameter tensors and persistent buffer tensors.

The causal masks in this model are buffers, so `state_dict()` contains more than learned parameters.

A **model configuration** records the arguments needed to recreate the architecture.

A **training configuration** documents hyperparameters but does not contain optimizer moments, the current step, random-number-generator states, or data-loader state.

An **inference package** is enough to encode prompts and run the trained model.

A **resumable checkpoint** needs additional training state for an exact continuation.

## The compatibility invariant

Token ID meanings are part of the model contract.

If one tokenizer maps `'A'` to `7` and another maps it to `27`, the same prompt selects different embedding rows.

Vocabulary size alone cannot detect that semantic mismatch.

Architecture fields must also match tensor shapes and state-dictionary keys.

## Define a versioned character tokenizer

The serializer stores only `token_to_id` because `id_to_token` can be reconstructed without JSON integer-key conversion.

The loader rejects missing format metadata, non-integer IDs, duplicate IDs, and gaps in the ID range.

In [1]:
class SaveableCharacterTokenizer:
    token_to_id: dict[str, int]
    id_to_token: dict[int, str]
    is_trained: bool

    def __init__(self) -> None:
        self.token_to_id = {}
        self.id_to_token = {}
        self.is_trained = False

    def train(self, text: str) -> None:
        if not text:
            raise ValueError("Training text must not be empty.")
        self.token_to_id = {
            token: token_id for token_id, token in enumerate(sorted(set(text)))
        }
        self.id_to_token = {
            token_id: token for token, token_id in self.token_to_id.items()
        }
        self.is_trained = True

    def _check_trained(self) -> None:
        if not self.is_trained:
            raise RuntimeError("Tokenizer must be trained before use.")

    @property
    def vocabulary_size(self) -> int:
        self._check_trained()
        return len(self.token_to_id)

    def encode(self, text: str) -> list[int]:
        self._check_trained()
        unknown_tokens = sorted(set(text) - set(self.token_to_id))
        if unknown_tokens:
            raise ValueError(f"Unknown tokens: {unknown_tokens!r}.")
        return [self.token_to_id[token] for token in text]

    def decode(self, token_ids: list[int]) -> str:
        self._check_trained()
        return "".join(self.id_to_token[token_id] for token_id in token_ids)

    def to_dict(self) -> dict[str, object]:
        self._check_trained()
        return {
            "format_version": 1,
            "tokenizer_type": "character",
            "token_to_id": self.token_to_id,
        }

    @classmethod
    def from_dict(
        cls,
        tokenizer_data: dict[str, object],
    ) -> "SaveableCharacterTokenizer":
        expected_keys = {"format_version", "tokenizer_type", "token_to_id"}
        if set(tokenizer_data) != expected_keys:
            raise ValueError("Tokenizer JSON has missing or unexpected fields.")
        if tokenizer_data["format_version"] != 1:
            raise ValueError("Unsupported tokenizer format version.")
        if tokenizer_data["tokenizer_type"] != "character":
            raise ValueError("Expected a character tokenizer.")
        raw_mapping = tokenizer_data["token_to_id"]
        if not isinstance(raw_mapping, dict) or not raw_mapping:
            raise TypeError("token_to_id must be a non-empty dictionary.")

        token_to_id: dict[str, int] = {}
        for token, token_id in raw_mapping.items():
            if not isinstance(token, str) or type(token_id) is not int:
                raise TypeError("Tokenizer entries must map strings to integer IDs.")
            token_to_id[token] = token_id
        expected_ids = list(range(len(token_to_id)))
        if sorted(token_to_id.values()) != expected_ids:
            raise ValueError("Tokenizer IDs must be unique and contiguous from zero.")

        tokenizer = cls()
        tokenizer.token_to_id = token_to_id
        tokenizer.id_to_token = {
            token_id: token for token, token_id in token_to_id.items()
        }
        tokenizer.is_trained = True
        return tokenizer

## Test tokenizer serialization in memory

An in-memory round trip isolates tokenizer schema correctness before file I/O is introduced.

In [2]:
training_text = (
    "Alice saw the rabbit. "
    "The rabbit saw Alice. "
    "Alice followed the rabbit. "
    "The rabbit was late. "
) * 40

tokenizer = SaveableCharacterTokenizer()
tokenizer.train(training_text)
restored_in_memory_tokenizer = SaveableCharacterTokenizer.from_dict(tokenizer.to_dict())
probe_text = "Alice"

print("Vocabulary size:", tokenizer.vocabulary_size)
print("Probe IDs:", tokenizer.encode(probe_text))
print("Round trip:", restored_in_memory_tokenizer.decode(tokenizer.encode(probe_text)))

assert restored_in_memory_tokenizer.token_to_id == tokenizer.token_to_id
assert restored_in_memory_tokenizer.encode(probe_text) == tokenizer.encode(probe_text)

Vocabulary size: 18
Probe IDs: [2, 12, 11, 6, 8]
Round trip: Alice


## Define the same TinyGPT architecture

The model uses the `token_embedding` and `position_embedding` names established in recent chapters.

Its state dictionary will therefore depend on these class and attribute names as well as tensor shapes.

In [3]:
import math  # noqa: I001
import torch


class MultiHeadCausalSelfAttention(torch.nn.Module):
    embedding_dimension: int
    number_of_attention_heads: int
    head_size: int

    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        if embedding_dimension % number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        self.embedding_dimension = embedding_dimension
        self.number_of_attention_heads = number_of_attention_heads
        self.head_size = embedding_dimension // number_of_attention_heads
        self.query_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.key_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.value_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension, bias=False
        )
        self.output_projection = torch.nn.Linear(
            embedding_dimension, embedding_dimension
        )
        self.attention_dropout = torch.nn.Dropout(dropout_rate)
        self.output_dropout = torch.nn.Dropout(dropout_rate)
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_length, context_length, dtype=torch.bool)),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        batch_size, sequence_length, _ = input_values.shape

        def split_heads(values: torch.Tensor) -> torch.Tensor:
            return values.view(
                batch_size,
                sequence_length,
                self.number_of_attention_heads,
                self.head_size,
            ).transpose(1, 2)

        queries = split_heads(self.query_projection(input_values))
        keys = split_heads(self.key_projection(input_values))
        values = split_heads(self.value_projection(input_values))
        attention_scores = queries @ keys.transpose(-2, -1)
        attention_scores = attention_scores / math.sqrt(self.head_size)
        causal_mask = self.causal_mask[:sequence_length, :sequence_length]
        attention_scores = attention_scores.masked_fill(~causal_mask, float("-inf"))
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended_values = self.attention_dropout(attention_weights) @ values
        concatenated_values = (
            attended_values.transpose(1, 2)
            .contiguous()
            .view(batch_size, sequence_length, self.embedding_dimension)
        )
        projected_values = self.output_projection(concatenated_values)
        output_values: torch.Tensor = self.output_dropout(projected_values)
        return output_values

In [4]:
class FeedForwardNetwork(torch.nn.Module):
    def __init__(self, embedding_dimension: int, dropout_rate: float) -> None:
        super().__init__()
        self.network = torch.nn.Sequential(
            torch.nn.Linear(embedding_dimension, 4 * embedding_dimension),
            torch.nn.ReLU(),
            torch.nn.Linear(4 * embedding_dimension, embedding_dimension),
            torch.nn.Dropout(dropout_rate),
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        output_values: torch.Tensor = self.network(input_values)
        return output_values


class TransformerBlock(torch.nn.Module):
    def __init__(
        self,
        embedding_dimension: int,
        number_of_attention_heads: int,
        context_length: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.attention_norm = torch.nn.LayerNorm(embedding_dimension)
        self.attention = MultiHeadCausalSelfAttention(
            embedding_dimension=embedding_dimension,
            number_of_attention_heads=number_of_attention_heads,
            context_length=context_length,
            dropout_rate=dropout_rate,
        )
        self.feedforward_norm = torch.nn.LayerNorm(embedding_dimension)
        self.feedforward = FeedForwardNetwork(
            embedding_dimension=embedding_dimension,
            dropout_rate=dropout_rate,
        )

    def forward(self, input_values: torch.Tensor) -> torch.Tensor:
        attention_branch = self.attention(self.attention_norm(input_values))
        values_after_attention = input_values + attention_branch
        feedforward_branch = self.feedforward(
            self.feedforward_norm(values_after_attention)
        )
        output_values: torch.Tensor = values_after_attention + feedforward_branch
        return output_values


class TinyGPT(torch.nn.Module):
    vocabulary_size: int
    context_length: int

    def __init__(
        self,
        vocabulary_size: int,
        context_length: int,
        embedding_dimension: int,
        number_of_attention_heads: int,
        number_of_transformer_blocks: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.vocabulary_size = vocabulary_size
        self.context_length = context_length
        self.token_embedding = torch.nn.Embedding(vocabulary_size, embedding_dimension)
        self.position_embedding = torch.nn.Embedding(
            context_length, embedding_dimension
        )
        self.embedding_dropout = torch.nn.Dropout(dropout_rate)
        self.transformer_blocks = torch.nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dimension=embedding_dimension,
                    number_of_attention_heads=number_of_attention_heads,
                    context_length=context_length,
                    dropout_rate=dropout_rate,
                )
                for _ in range(number_of_transformer_blocks)
            ]
        )
        self.final_norm = torch.nn.LayerNorm(embedding_dimension)
        self.output_layer = torch.nn.Linear(embedding_dimension, vocabulary_size)

    def forward(
        self,
        input_token_ids: torch.Tensor,
        target_token_ids: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        if input_token_ids.ndim != 2:
            raise ValueError("Inputs must have shape [batch, time].")
        batch_size, sequence_length = input_token_ids.shape
        if sequence_length > self.context_length:
            raise ValueError("Input exceeds model context length.")
        position_ids = torch.arange(sequence_length, device=input_token_ids.device)
        hidden_values = self.token_embedding(input_token_ids)
        hidden_values = hidden_values + self.position_embedding(position_ids)
        hidden_values = self.embedding_dropout(hidden_values)
        hidden_values = self.transformer_blocks(hidden_values)
        logits = self.output_layer(self.final_norm(hidden_values))
        loss = None
        if target_token_ids is not None:
            if target_token_ids.shape != input_token_ids.shape:
                raise ValueError("Targets must have the same shape as inputs.")
            loss = torch.nn.functional.cross_entropy(
                logits.reshape(batch_size * sequence_length, self.vocabulary_size),
                target_token_ids.reshape(batch_size * sequence_length),
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        input_token_ids: torch.Tensor,
        number_of_new_tokens: int,
        generator: torch.Generator,
        temperature: float = 1.0,
        top_k: int | None = None,
    ) -> torch.Tensor:
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        generated_ids = input_token_ids
        for _ in range(number_of_new_tokens):
            model_input = generated_ids[:, -self.context_length :]
            logits, _ = self(model_input)
            next_logits = logits[:, -1] / temperature
            if top_k is not None:
                effective_top_k = min(top_k, self.vocabulary_size)
                top_values, _ = torch.topk(next_logits, effective_top_k)
                next_logits = next_logits.masked_fill(
                    next_logits < top_values[:, -1, None], float("-inf")
                )
            probabilities = torch.softmax(next_logits, dim=-1)
            next_ids = torch.multinomial(
                probabilities, num_samples=1, generator=generator
            )
            generated_ids = torch.cat([generated_ids, next_ids], dim=1)
        return generated_ids

## Define validated configuration objects

Version and architecture fields prevent a loader from silently guessing which code should interpret the saved values.

The parsers reject booleans where integers are required because JSON booleans are Python `bool`, which is a subclass of `int`.

In [5]:
from dataclasses import dataclass


def require_int(
    data: dict[str, object],
    key: str,
    minimum: int,
) -> int:
    value = data.get(key)
    if type(value) is not int or value < minimum:
        raise ValueError(f"{key} must be an integer of at least {minimum}.")
    return value


def require_float(
    data: dict[str, object],
    key: str,
    minimum: float,
    maximum: float | None = None,
) -> float:
    value = data.get(key)
    if type(value) is not float or value < minimum:
        raise ValueError(f"{key} must be a floating-point value of at least {minimum}.")
    if maximum is not None and value >= maximum:
        raise ValueError(f"{key} must be less than {maximum}.")
    return value


@dataclass(frozen=True)
class ModelConfig:
    format_version: int
    architecture: str
    vocabulary_size: int
    context_length: int
    embedding_dimension: int
    number_of_attention_heads: int
    number_of_transformer_blocks: int
    dropout_rate: float

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "architecture": self.architecture,
            "vocabulary_size": self.vocabulary_size,
            "context_length": self.context_length,
            "embedding_dimension": self.embedding_dimension,
            "number_of_attention_heads": self.number_of_attention_heads,
            "number_of_transformer_blocks": self.number_of_transformer_blocks,
            "dropout_rate": self.dropout_rate,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "ModelConfig":
        expected_keys = {
            "format_version",
            "architecture",
            "vocabulary_size",
            "context_length",
            "embedding_dimension",
            "number_of_attention_heads",
            "number_of_transformer_blocks",
            "dropout_rate",
        }
        if set(data) != expected_keys:
            raise ValueError("Model config has missing or unexpected fields.")
        if data["format_version"] != 1 or data["architecture"] != "TinyGPT-v1":
            raise ValueError("Unsupported model config format or architecture.")
        config = cls(
            format_version=1,
            architecture="TinyGPT-v1",
            vocabulary_size=require_int(data, "vocabulary_size", 1),
            context_length=require_int(data, "context_length", 1),
            embedding_dimension=require_int(data, "embedding_dimension", 1),
            number_of_attention_heads=require_int(data, "number_of_attention_heads", 1),
            number_of_transformer_blocks=require_int(
                data, "number_of_transformer_blocks", 1
            ),
            dropout_rate=require_float(data, "dropout_rate", 0.0, 1.0),
        )
        if config.embedding_dimension % config.number_of_attention_heads != 0:
            raise ValueError("Attention heads must divide the embedding dimension.")
        return config


@dataclass(frozen=True)
class TrainingConfig:
    format_version: int
    batch_size: int
    learning_rate: float
    training_steps: int
    weight_decay: float
    max_gradient_norm: float
    model_seed: int
    training_seed: int
    dataset_description: str

    def to_dict(self) -> dict[str, object]:
        return {
            "format_version": self.format_version,
            "batch_size": self.batch_size,
            "learning_rate": self.learning_rate,
            "training_steps": self.training_steps,
            "weight_decay": self.weight_decay,
            "max_gradient_norm": self.max_gradient_norm,
            "model_seed": self.model_seed,
            "training_seed": self.training_seed,
            "dataset_description": self.dataset_description,
        }

    @classmethod
    def from_dict(cls, data: dict[str, object]) -> "TrainingConfig":
        expected_keys = {
            "format_version",
            "batch_size",
            "learning_rate",
            "training_steps",
            "weight_decay",
            "max_gradient_norm",
            "model_seed",
            "training_seed",
            "dataset_description",
        }
        if set(data) != expected_keys or data["format_version"] != 1:
            raise ValueError("Unsupported training config schema.")
        description = data["dataset_description"]
        if not isinstance(description, str) or not description:
            raise ValueError("dataset_description must be a non-empty string.")
        learning_rate = require_float(data, "learning_rate", 0.0)
        max_gradient_norm = require_float(data, "max_gradient_norm", 0.0)
        if learning_rate == 0.0 or max_gradient_norm == 0.0:
            raise ValueError("learning_rate and max_gradient_norm must be positive.")
        return cls(
            format_version=1,
            batch_size=require_int(data, "batch_size", 1),
            learning_rate=learning_rate,
            training_steps=require_int(data, "training_steps", 1),
            weight_decay=require_float(data, "weight_decay", 0.0),
            max_gradient_norm=max_gradient_norm,
            model_seed=require_int(data, "model_seed", 0),
            training_seed=require_int(data, "training_seed", 0),
            dataset_description=description,
        )

## Create matching configs and model factory

One factory is used for both the original model and the fresh loaded model, reducing opportunities for architecture drift.

In [6]:
model_config = ModelConfig(
    format_version=1,
    architecture="TinyGPT-v1",
    vocabulary_size=tokenizer.vocabulary_size,
    context_length=64,
    embedding_dimension=64,
    number_of_attention_heads=4,
    number_of_transformer_blocks=2,
    dropout_rate=0.1,
)
training_config = TrainingConfig(
    format_version=1,
    batch_size=8,
    learning_rate=0.0003,
    training_steps=300,
    weight_decay=0.01,
    max_gradient_norm=1.0,
    model_seed=8801,
    training_seed=8802,
    dataset_description="Four fixed Alice sentences repeated 40 times.",
)


def create_model(config: ModelConfig) -> TinyGPT:
    return TinyGPT(
        vocabulary_size=config.vocabulary_size,
        context_length=config.context_length,
        embedding_dimension=config.embedding_dimension,
        number_of_attention_heads=config.number_of_attention_heads,
        number_of_transformer_blocks=config.number_of_transformer_blocks,
        dropout_rate=config.dropout_rate,
    )


torch.manual_seed(training_config.model_seed)
model = create_model(model_config)
print(
    "Model vocabulary matches tokenizer:",
    model.vocabulary_size == tokenizer.vocabulary_size,
)
assert model.vocabulary_size == tokenizer.vocabulary_size

Model vocabulary matches tokenizer: True


## Train before saving

Saving a randomly initialized model would test file I/O but not the promised trained-model workflow.

This focused loop performs exactly 300 AdamW updates and prints deterministic checkpoints.

In [7]:
def get_training_batch(
    token_ids: torch.Tensor,
    batch_size: int,
    context_length: int,
    generator: torch.Generator,
) -> tuple[torch.Tensor, torch.Tensor]:
    valid_starts = token_ids.numel() - context_length
    if valid_starts < 1:
        raise ValueError("Token stream must be longer than context_length.")
    starts = torch.randint(0, valid_starts, (batch_size,), generator=generator)
    offsets = torch.arange(context_length)
    indexes = starts[:, None] + offsets[None, :]
    return token_ids[indexes], token_ids[indexes + 1]


all_token_ids = torch.tensor(tokenizer.encode(training_text), dtype=torch.long)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=training_config.learning_rate,
    weight_decay=training_config.weight_decay,
)
batch_generator = torch.Generator().manual_seed(training_config.training_seed)
torch.manual_seed(training_config.training_seed)
model.train()
checkpoint_losses: list[float] = []

for step in range(1, training_config.training_steps + 1):
    inputs, targets = get_training_batch(
        all_token_ids,
        training_config.batch_size,
        model_config.context_length,
        batch_generator,
    )
    _, loss = model(inputs, targets)
    if loss is None or not torch.isfinite(loss):
        raise RuntimeError("Training loss is non-finite.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=training_config.max_gradient_norm,
        error_if_nonfinite=True,
    )
    optimizer.step()
    if step == 1 or step % 100 == 0:
        checkpoint_losses.append(float(loss.item()))
        print(f"step {step:>3} | batch loss {loss.item():.4f}")

assert len(checkpoint_losses) == 4

step   1 | batch loss 2.9283


step 100 | batch loss 1.2853


step 200 | batch loss 0.9810


step 300 | batch loss 0.7863


The batch loss falls during this short run, so the state saved below contains learned updates rather than only initialization.

## Create temporary artifact paths

The example uses a temporary directory so executing the instructional notebook does not leave generated files in the repository.

A real project would replace it with a persistent, versioned artifact directory.

In [8]:
import tempfile  # noqa: I001
from pathlib import Path


temporary_directory = tempfile.TemporaryDirectory()
artifact_directory = Path(temporary_directory.name)
model_state_path = artifact_directory / "model_state.pt"
tokenizer_path = artifact_directory / "tokenizer.json"
model_config_path = artifact_directory / "model_config.json"
training_config_path = artifact_directory / "training_config.json"

print("Artifact filenames:")
for path in [
    model_state_path,
    tokenizer_path,
    model_config_path,
    training_config_path,
]:
    print("-", path.name)

Artifact filenames:
- model_state.pt
- tokenizer.json
- model_config.json
- training_config.json


## Save JSON and model state

JSON is human-readable and appropriate for the tokenizer and plain configuration values.

`torch.save(model.state_dict(), path)` preserves tensor dtypes, shapes, names, and persistent buffers.

In [9]:
import json  # noqa: I001
from collections.abc import Mapping


def save_json(data: Mapping[str, object], path: Path) -> None:
    serialized = json.dumps(
        dict(data),
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    path.write_text(serialized + "\n", encoding="utf-8")


torch.save(model.state_dict(), model_state_path)
save_json(tokenizer.to_dict(), tokenizer_path)
save_json(model_config.to_dict(), model_config_path)
save_json(training_config.to_dict(), training_config_path)

saved_filenames = sorted(path.name for path in artifact_directory.iterdir())
print("Saved files:", saved_filenames)
assert saved_filenames == [
    "model_config.json",
    "model_state.pt",
    "tokenizer.json",
    "training_config.json",
]

Saved files: ['model_config.json', 'model_state.pt', 'tokenizer.json', 'training_config.json']


## Inspect what was saved

The JSON keys are readable without importing PyTorch.

State-dictionary keys expose the coupling between saved tensor names and model code.

In [10]:
saved_tokenizer_document = tokenizer.to_dict()
saved_mapping = saved_tokenizer_document["token_to_id"]
if not isinstance(saved_mapping, dict):
    raise RuntimeError("Saved tokenizer mapping has the wrong type.")
print("Tokenizer JSON keys:", sorted(saved_tokenizer_document))
print("Tokenizer mapping entries:", len(saved_mapping))
print("First eight mappings:", list(saved_mapping.items())[:8])
print("Model config JSON:")
print(model_config_path.read_text(encoding="utf-8"))

state_dictionary = model.state_dict()
print("State entries:", len(state_dictionary))
print("First eight state keys:")
for state_key in list(state_dictionary)[:8]:
    print("-", state_key, tuple(state_dictionary[state_key].shape))

assert any(state_key.endswith("causal_mask") for state_key in state_dictionary)

Tokenizer JSON keys: ['format_version', 'token_to_id', 'tokenizer_type']
Tokenizer mapping entries: 18
First eight mappings: [(' ', 0), ('.', 1), ('A', 2), ('T', 3), ('a', 4), ('b', 5), ('c', 6), ('d', 7)]
Model config JSON:
{
  "architecture": "TinyGPT-v1",
  "context_length": 64,
  "dropout_rate": 0.1,
  "embedding_dimension": 64,
  "format_version": 1,
  "number_of_attention_heads": 4,
  "number_of_transformer_blocks": 2,
  "vocabulary_size": 18
}

State entries: 34
First eight state keys:
- token_embedding.weight (18, 64)
- position_embedding.weight (64, 64)
- transformer_blocks.0.attention_norm.weight (64,)
- transformer_blocks.0.attention_norm.bias (64,)
- transformer_blocks.0.attention.causal_mask (64, 64)
- transformer_blocks.0.attention.query_projection.weight (64, 64)
- transformer_blocks.0.attention.key_projection.weight (64, 64)
- transformer_blocks.0.attention.value_projection.weight (64, 64)


The causal-mask entry confirms that persistent buffers accompany learned weights in the state dictionary.

## Load and validate JSON

JSON decoding returns general Python objects, so typed constructors must validate every field before the values reach model or tokenizer code.

In [11]:
from typing import cast


def load_json_object(path: Path) -> dict[str, object]:
    loaded_value = cast(object, json.loads(path.read_text(encoding="utf-8")))
    if not isinstance(loaded_value, dict):
        raise TypeError("Expected a JSON object at the top level.")
    result: dict[str, object] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str):
            raise TypeError("JSON object keys must be strings.")
        result[key] = value
    return result


loaded_tokenizer = SaveableCharacterTokenizer.from_dict(
    load_json_object(tokenizer_path)
)
loaded_model_config = ModelConfig.from_dict(load_json_object(model_config_path))
loaded_training_config = TrainingConfig.from_dict(
    load_json_object(training_config_path)
)

print("Loaded tokenizer vocabulary:", loaded_tokenizer.vocabulary_size)
print("Loaded architecture:", loaded_model_config.architecture)
print("Loaded training steps:", loaded_training_config.training_steps)

if loaded_tokenizer.vocabulary_size != loaded_model_config.vocabulary_size:
    raise RuntimeError("Tokenizer and model vocabularies do not match.")

Loaded tokenizer vocabulary: 18
Loaded architecture: TinyGPT-v1
Loaded training steps: 300


## Load tensor state safely

`map_location='cpu'` makes device placement explicit.

`weights_only=True` restricts PyTorch's unpickler to tensors, primitive values, dictionaries, and explicitly allowed types.

Only load artifacts from trusted sources because serialization formats and surrounding files can still be malformed or incompatible.

In [12]:
def load_tensor_state_dict(path: Path) -> dict[str, torch.Tensor]:
    loaded_value: object = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )
    if not isinstance(loaded_value, dict):
        raise TypeError("Model state must be a dictionary.")
    tensor_state: dict[str, torch.Tensor] = {}
    for key, value in loaded_value.items():
        if not isinstance(key, str) or not isinstance(value, torch.Tensor):
            raise TypeError("Model state must map string names to tensors.")
        tensor_state[key] = value
    return tensor_state


loaded_model = create_model(loaded_model_config)
loaded_state_dictionary = load_tensor_state_dict(model_state_path)
incompatible_keys = loaded_model.load_state_dict(loaded_state_dictionary, strict=True)
loaded_model.eval()

print("Missing keys:", incompatible_keys.missing_keys)
print("Unexpected keys:", incompatible_keys.unexpected_keys)
assert not incompatible_keys.missing_keys
assert not incompatible_keys.unexpected_keys

Missing keys: []
Unexpected keys: []


## Verify the complete inference round trip

The restored tokenizer must produce identical IDs, every loaded state tensor must match exactly, and evaluation-mode logits must be bitwise equal on CPU.

In [13]:
model.eval()
prompt = "Alice"
original_prompt_ids = tokenizer.encode(prompt)
loaded_prompt_ids = loaded_tokenizer.encode(prompt)
prompt_tensor = torch.tensor([loaded_prompt_ids], dtype=torch.long)

with torch.no_grad():
    original_logits, _ = model(prompt_tensor)
    loaded_logits, _ = loaded_model(prompt_tensor)

state_tensors_match = all(
    torch.equal(state_dictionary[key], loaded_state_dictionary[key])
    for key in state_dictionary
)
print("Tokenizer IDs match:", original_prompt_ids == loaded_prompt_ids)
print("State tensors match exactly:", state_tensors_match)
print("Logits match exactly:", torch.equal(original_logits, loaded_logits))

assert original_prompt_ids == loaded_prompt_ids
assert state_tensors_match
assert torch.equal(original_logits, loaded_logits)

Tokenizer IDs match: True
State tensors match exactly: True
Logits match exactly: True


## Generate from both systems with the same seed

Explicit generators isolate sampling randomness and let the round trip require identical token sequences, not merely plausible text.

In [14]:
def generate_text(
    model: TinyGPT,
    tokenizer: SaveableCharacterTokenizer,
    prompt: str,
    number_of_new_tokens: int,
    seed: int,
) -> str:
    model.eval()
    input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long)
    generator = torch.Generator().manual_seed(seed)
    generated_ids = model.generate(
        input_ids,
        number_of_new_tokens=number_of_new_tokens,
        generator=generator,
        temperature=1.0,
        top_k=10,
    )
    return tokenizer.decode(generated_ids[0].tolist())


original_sample = generate_text(model, tokenizer, "Alice", 100, seed=8803)
loaded_sample = generate_text(loaded_model, loaded_tokenizer, "Alice", 100, seed=8803)

print("Loaded sample:")
print(loaded_sample)
print("Samples match exactly:", original_sample == loaded_sample)
assert original_sample == loaded_sample

Loaded sample:
Alice the Aliced rabit raw rabbbbice raAliche rasate sabbat. Allice fololllolloliced follololllithe rloll
Samples match exactly: True


Identical seeded generation proves that architecture, tensors, token meanings, evaluation mode, and sampling inputs were restored consistently for this environment.

It does not guarantee compatibility with arbitrary future code or library versions.

## Demonstrate tokenizer mismatch

A reversed but still contiguous ID mapping has the same vocabulary size and can encode the prompt, yet it selects the wrong embedding rows.

In [15]:
reversed_mapping = {
    token: tokenizer.vocabulary_size - 1 - token_id
    for token, token_id in tokenizer.token_to_id.items()
}
wrong_tokenizer = SaveableCharacterTokenizer.from_dict(
    {
        "format_version": 1,
        "tokenizer_type": "character",
        "token_to_id": reversed_mapping,
    }
)
correct_ids = loaded_tokenizer.encode(prompt)
wrong_ids = wrong_tokenizer.encode(prompt)

print("Correct IDs:", correct_ids)
print("Wrong IDs:  ", wrong_ids)
print(
    "Vocabulary sizes match:",
    wrong_tokenizer.vocabulary_size == loaded_model.vocabulary_size,
)
print("Prompt IDs match:", correct_ids == wrong_ids)

assert wrong_tokenizer.vocabulary_size == loaded_model.vocabulary_size
assert correct_ids != wrong_ids

Correct IDs: [2, 12, 11, 6, 8]
Wrong IDs:   [15, 5, 6, 11, 9]
Vocabulary sizes match: True
Prompt IDs match: False


This mismatch passes a vocabulary-size check, which is why the exact tokenizer mapping must travel with the weights.

## Inference package versus resumable checkpoint

The four files demonstrated above are sufficient for inference in compatible code.

An exact training continuation should additionally save at least:

- `optimizer.state_dict()` for AdamW moments and parameter-group state;

- the completed optimizer-update count;

- CPU and accelerator random-number-generator states;

- batch sampler or data-loader position and state; and

- learning-rate-scheduler state when a scheduler is used.

Training configuration documents settings but cannot replace this dynamic state.

## Versioning and code changes

Renaming `token_embedding` or changing block structure changes state-dictionary keys.

Changing vocabulary size, embedding dimension, layer count, or context length changes tensor shapes or buffer shapes.

Real projects should version model code, configuration schemas, tokenizer schemas, dataset provenance, and dependency versions alongside artifacts.

Migration code is preferable to silently coercing an old package into a new architecture.

## Clean up the temporary package

All validation occurs before cleanup, and the final assertion confirms that notebook execution leaves no artifact directory behind.

In [16]:
temporary_directory.cleanup()
print("Temporary artifact directory removed:", not artifact_directory.exists())
assert not artifact_directory.exists()

Temporary artifact directory removed: True


## Verify chapter contracts

The final checks cover the trained update count, schema round trips, model state, inference outputs, and tokenizer mismatch demonstration.

In [17]:
assert training_config.training_steps == 300
assert checkpoint_losses[-1] < checkpoint_losses[0]
assert loaded_model_config == model_config
assert loaded_training_config == training_config
assert loaded_tokenizer.token_to_id == tokenizer.token_to_id
assert original_sample == loaded_sample
assert correct_ids != wrong_ids
assert not artifact_directory.exists()
print("All save-and-load contracts passed.")

All save-and-load contracts passed.


## Common mistakes

- Saving only tensor state loses token meanings and architecture arguments.

- Calling every state-dictionary tensor a learned parameter ignores persistent buffers.

- Rebuilding a tokenizer from similar text can assign different IDs or subword rules.

- Checking only vocabulary size cannot detect a permuted token mapping.

- Passing unvalidated JSON values directly into constructors turns corrupt files into confusing downstream errors.

- Loading untrusted pickle-based artifacts without restrictions creates a security risk.

- Forgetting `map_location` can restore tensors to an unavailable device.

- Comparing models in training mode lets dropout hide a correct round trip.

- Treating training config as a resumable checkpoint omits optimizer, scheduler, RNG, and sampler state.

- Changing class attribute names or architecture can invalidate saved state keys and shapes.

## Takeaways

- A reusable GPT package needs compatible model code, model state, tokenizer data, and model configuration.

- Training configuration improves documentation but is not enough for exact resume.

- JSON schemas and contiguous tokenizer IDs should be validated during loading.

- `state_dict()` includes learned parameters and persistent buffers but not Python class code.

- `map_location='cpu'`, `weights_only=True`, and trusted artifact sources make tensor loading safer.

- Exact ID, tensor, logit, and seeded-generation checks provide a strong inference round-trip test.

- The exact tokenizer mapping is part of the model contract even when another tokenizer has the same vocabulary size.

## What comes next

An inference package restores a fixed model, while checkpointing during training must decide when to save and how to resume dynamic optimizer state.

The next checkpoint workflow can build on the validation and compatibility rules established here.